# Формирование отчета Word

Ноутбук собирает отчет `rgr2/report/РГР_2_отчет.docx` по данным, полученным в ноутбуках 01–03.

Перед запуском при необходимости укажите свои данные на титульном листе в переменных `STUDENT`, `COURSE`, `GROUP` ниже. Остальное (таблицы, рисунки, подписи, оглавление) формируется автоматически.

Оформление: Times New Roman 14 пт, межстрочный интервал 18 пт, поля 20/20/30/10 мм, заголовки прописными полужирным шрифтом по центру, каждый раздел с нового листа, подписи рисунков снизу, названия таблиц сверху слева.


In [1]:
from pathlib import Path
import re
import subprocess
import pandas as pd
import numpy as np
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
REPORT_DIR = BASE / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Данные для титульного листа
STUDENT = '[ФИО студента]'
COURSE = '[курс]'
GROUP = '[группа]'

# Загружаем результаты расчетов
zipf = pd.read_csv(DATA / 'zipf_summary.csv')
fw_summary = pd.read_csv(DATA / 'zipf2_function_words_summary.csv')
juilland_table = pd.read_csv(DATA / 'juilland_table_top10.csv')

print('Данные загружены.')
print(zipf[['key', 'dictionary', 'c_median']].to_string(index=False))


Данные загружены.
     key                   dictionary  c_median
  ru_raw    Русский, без лемматизации  591965.0
ru_lemma     Русский, с лемматизацией  656775.0
  en_raw Английский, без лемматизации  581872.0
en_lemma  Английский, с лемматизацией  567200.0


In [2]:
TOC_ENTRIES = [
    'ВВЕДЕНИЕ',
    'ЗАДАНИЕ 1. ЗАКОН ЦИПФА',
    'ЗАДАНИЕ 2. ЭМПИРИЧЕСКИЙ ЗАКОН ЦИПФА',
    'ЗАДАНИЕ 3. РАСЧЕТ КОЭФФИЦИЕНТА D ЖУЙАНА',
    'ЗАКЛЮЧЕНИЕ',
    'СПИСОК ИСПОЛЬЗОВАННЫХ ИСТОЧНИКОВ',
]


def fmt(x, digits=3):
    return f"{x:.{digits}f}".replace('.', ',')


def fmt_int(x):
    return f"{int(round(x)):,}".replace(',', ' ')


def set_run_font(run, size=14, bold=False, italic=False, name='Times New Roman'):
    run.bold = bold
    run.italic = italic
    run.font.size = Pt(size)
    run.font.name = name
    rpr = run._element.get_or_add_rPr()
    rfonts = rpr.find(qn('w:rFonts'))
    if rfonts is None:
        rfonts = OxmlElement('w:rFonts')
        rpr.append(rfonts)
    for attr in ('w:ascii', 'w:hAnsi', 'w:eastAsia', 'w:cs'):
        rfonts.set(qn(attr), name)


def add_para(doc, text='', size=14, bold=False, italic=False,
             align=WD_ALIGN_PARAGRAPH.JUSTIFY, indent=True,
             before=0, after=0, keep=False, line=18):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.first_line_indent = Cm(1.25) if indent else Cm(0)
    pf.line_spacing = Pt(line)
    pf.space_before = Pt(before)
    pf.space_after = Pt(after)
    pf.alignment = align
    pf.keep_with_next = keep
    if text:
        r = p.add_run(text)
        set_run_font(r, size=size, bold=bold, italic=italic)
    return p


def add_heading(doc, text, page_break=True):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.first_line_indent = Cm(0)
    pf.line_spacing = Pt(18)
    pf.space_before = Pt(0)
    pf.space_after = Pt(36)
    pf.alignment = WD_ALIGN_PARAGRAPH.CENTER
    pf.keep_with_next = True
    pf.page_break_before = page_break
    r = p.add_run(text.upper())
    set_run_font(r, size=16, bold=True)
    return p


def add_formula(doc, text):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.first_line_indent = Cm(0)
    pf.line_spacing = Pt(18)
    pf.space_before = Pt(6)
    pf.space_after = Pt(6)
    pf.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(text)
    set_run_font(r, size=14, italic=True)
    return p


def add_bullet(doc, text):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.first_line_indent = Cm(0)
    pf.left_indent = Cm(1.25)
    pf.line_spacing = Pt(18)
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)
    pf.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    r = p.add_run('— ' + text)
    set_run_font(r, size=14)
    return p


def add_table(doc, caption, headers, rows, widths=None, align=None, font_size=11,
              keep_together=None, page_break=False):
    cap = doc.add_paragraph()
    cf = cap.paragraph_format
    cf.first_line_indent = Cm(0)
    cf.line_spacing = Pt(18)
    cf.space_before = Pt(6)
    cf.space_after = Pt(3)
    cf.alignment = WD_ALIGN_PARAGRAPH.LEFT
    cf.keep_with_next = True
    cf.page_break_before = page_break
    r = cap.add_run(caption)
    set_run_font(r, size=12, bold=True)

    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Table Grid'
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False

    tr_pr = table.rows[0]._tr.get_or_add_trPr()
    tbl_header = OxmlElement('w:tblHeader')
    tbl_header.set(qn('w:val'), 'true')
    tr_pr.append(tbl_header)

    for i, h in enumerate(headers):
        cell = table.rows[0].cells[i]
        cell.text = ''
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.first_line_indent = Cm(0)
        p.paragraph_format.line_spacing = Pt(12)
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(0)
        rr = p.add_run(str(h))
        set_run_font(rr, size=font_size, bold=True)

    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            p = cells[i].paragraphs[0]
            p.paragraph_format.first_line_indent = Cm(0)
            p.paragraph_format.line_spacing = Pt(12)
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            if align is None:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            else:
                p.alignment = align[i]
            rr = p.add_run(str(value))
            set_run_font(rr, size=font_size)

    if widths:
        for i, w in enumerate(widths):
            table.columns[i].width = Cm(w)
        for row in table.rows:
            for i, w in enumerate(widths):
                row.cells[i].width = Cm(w)

        tbl_pr = table._tbl.tblPr
        tbl_w = tbl_pr.find(qn('w:tblW'))
        if tbl_w is None:
            tbl_w = OxmlElement('w:tblW')
            tbl_pr.append(tbl_w)
        tbl_w.set(qn('w:type'), 'dxa')
        tbl_w.set(qn('w:w'), str(int(Cm(sum(widths)).twips)))
        layout = OxmlElement('w:tblLayout')
        layout.set(qn('w:type'), 'fixed')
        tbl_pr.append(layout)

    for row in table.rows:
        tr_pr = row._tr.get_or_add_trPr()
        cant_split = OxmlElement('w:cantSplit')
        tr_pr.append(cant_split)

    if keep_together is None:
        keep_together = len(rows) <= 15
    if keep_together:
        for row in table.rows[:-1]:
            for cell in row.cells:
                for p in cell.paragraphs:
                    p.paragraph_format.keep_with_next = True
    return table


def add_figure(doc, path, caption, width_cm=16.5, page_break=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.first_line_indent = Cm(0)
    # Для абзаца с рисунком нельзя задавать точный межстрочный интервал:
    # иначе LibreOffice/Word обрезает изображение по высоте строки.
    pf.line_spacing = 1.0
    pf.space_before = Pt(6)
    pf.space_after = Pt(3)
    pf.alignment = WD_ALIGN_PARAGRAPH.CENTER
    pf.keep_together = True
    pf.keep_with_next = True
    pf.page_break_before = page_break
    r = p.add_run()
    r.add_picture(str(path), width=Cm(width_cm))

    cap = doc.add_paragraph()
    cf = cap.paragraph_format
    cf.first_line_indent = Cm(0)
    cf.line_spacing = Pt(18)
    cf.space_before = Pt(0)
    cf.space_after = Pt(18)
    cf.alignment = WD_ALIGN_PARAGRAPH.CENTER
    cf.keep_together = True
    rr = cap.add_run(caption)
    set_run_font(rr, size=12, bold=True)
    return p


def add_page_number_footer(section):
    footer = section.footer
    p = footer.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.first_line_indent = Cm(0)
    run = p.add_run()
    fld_char = OxmlElement('w:fldChar')
    fld_char.set(qn('w:fldCharType'), 'begin')
    instr = OxmlElement('w:instrText')
    instr.set(qn('xml:space'), 'preserve')
    instr.text = ' PAGE '
    fld_end = OxmlElement('w:fldChar')
    fld_end.set(qn('w:fldCharType'), 'end')
    run._r.append(fld_char)
    run._r.append(instr)
    run._r.append(fld_end)
    set_run_font(run, size=12)


def setup_document():
    doc = Document()
    section = doc.sections[0]
    section.page_width = Cm(21.0)
    section.page_height = Cm(29.7)
    section.top_margin = Cm(2.0)
    section.bottom_margin = Cm(2.0)
    section.left_margin = Cm(3.0)
    section.right_margin = Cm(1.0)
    section.different_first_page_header_footer = True
    add_page_number_footer(section)

    style = doc.styles['Normal']
    style.font.name = 'Times New Roman'
    style.font.size = Pt(14)
    rpr = style.element.get_or_add_rPr()
    rfonts = rpr.find(qn('w:rFonts'))
    if rfonts is None:
        rfonts = OxmlElement('w:rFonts')
        rpr.append(rfonts)
    for attr in ('w:ascii', 'w:hAnsi', 'w:eastAsia', 'w:cs'):
        rfonts.set(qn(attr), 'Times New Roman')
    pf = style.paragraph_format
    pf.line_spacing = Pt(18)
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)
    pf.first_line_indent = Cm(1.25)
    pf.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return doc


def add_title_page(doc):
    def centered(text, **kwargs):
        p = add_para(doc, text, align=WD_ALIGN_PARAGRAPH.CENTER, indent=False, **kwargs)
        # Титульный лист центрируем по центру страницы, а не по центру
        # асимметричного текстового блока (поля 30/10 мм).
        p.paragraph_format.left_indent = Cm(0)
        p.paragraph_format.right_indent = Cm(2.0)
        return p

    centered('Белорусский государственный университет')
    centered('Факультет прикладной математики и информатики')
    add_para(doc, '', indent=False)
    centered('Кафедра информационных систем управления')
    add_para(doc, '', indent=False, before=60)
    centered('РАСЧЕТНО-ГРАФИЧЕСКАЯ РАБОТА № 2', size=16, bold=True, before=30)
    centered('Статистические исследования на базе частотных словарей', size=14, before=12)
    centered('русского и английского языков', size=14)
    add_para(doc, '', indent=False, before=100)
    for line in ['Выполнил:', f'студент {COURSE} курса {GROUP} группы', STUDENT]:
        p = add_para(doc, line, align=WD_ALIGN_PARAGRAPH.LEFT, indent=False)
        p.paragraph_format.left_indent = Cm(7.5)
    add_para(doc, '', indent=False, before=150)
    centered('Минск, 2026')


def add_toc(doc, pages=None):
    pages = pages or {}
    add_heading(doc, 'СОДЕРЖАНИЕ', page_break=True)
    for title in TOC_ENTRIES:
        p = doc.add_paragraph()
        pf = p.paragraph_format
        pf.first_line_indent = Cm(0)
        pf.line_spacing = Pt(18)
        pf.space_before = Pt(0)
        pf.space_after = Pt(0)
        pf.alignment = WD_ALIGN_PARAGRAPH.LEFT
        pf.tab_stops.add_tab_stop(Cm(16.5), WD_TAB_ALIGNMENT.RIGHT, WD_TAB_LEADER.DOTS)
        page = str(pages.get(title, '00'))
        r = p.add_run(title + '\t' + page)
        set_run_font(r, size=14)


In [3]:
def build_report(path, pages=None):
    doc = setup_document()
    add_title_page(doc)
    add_toc(doc, pages)

    # ---------------- ВВЕДЕНИЕ ----------------
    add_heading(doc, 'ВВЕДЕНИЕ', page_break=True)
    add_para(doc,
        'Цель работы — статистическое исследование частотных словарей, построенных по корпусу '
        'русской и английской Википедии. Исследование включает проверку закона Ципфа, проверку '
        'эмпирического закона Ципфа для служебных слов и расчет коэффициента D Жуйана.')
    add_para(doc,
        'Исходные данные — четыре частотных словаря проекта nlp_uni: русский и английский, каждый '
        'в двух вариантах — без лемматизации и с лемматизацией. Словари содержат абсолютные частоты '
        'слов (словоформ или лемм), общее число токенов и число уникальных единиц. Объем каждого '
        'словаря — около 5 млн словоупотреблений.')
    add_para(doc,
        'Для расчетов использовались язык программирования Python, библиотеки pandas, NumPy, '
        'SciPy и Matplotlib. Все вычисления и графики воспроизводятся в ноутбуках '
        '01_zipf_law.ipynb, 02_zipf_empirical_function_words.ipynb и 03_juilland.ipynb.')
    add_para(doc,
        'Отчет состоит из трех заданий. В первом задании проверяется основной закон Ципфа, во втором — '
        'его эмпирический вариант для служебных слов, в третьем — рассчитывается коэффициент '
        'Жуйана для наиболее частых слов.')

    # ---------------- ЗАДАНИЕ 1 ----------------
    add_heading(doc, 'ЗАДАНИЕ 1. ЗАКОН ЦИПФА', page_break=True)
    add_para(doc,
        'Закон Ципфа утверждает, что произведение частоты слова f на его ранг r в частотном списке '
        'есть приблизительно постоянная величина:')
    add_formula(doc, 'f · r = c,')
    add_para(doc,
        'где f — частота встречаемости слова в тексте; r — ранг слова в частотном списке; '
        'c — эмпирическая постоянная (коэффициент Ципфа).')
    add_para(doc,
        'Ранг присваивался словам по убыванию частоты. Для каждого слова вычислялось произведение '
        'f · r, а коэффициент c оценивался как медиана этого произведения в диапазоне рангов '
        '100–5000. Характеристики словарей приведены в таблице 1.')

    rows = []
    for _, row in zipf.iterrows():
        lang = 'Русский' if row['key'].startswith('ru') else 'Английский'
        lemma = 'Да' if row['key'].endswith('lemma') else 'Нет'
        rows.append([row['dictionary'], lang, lemma,
                     f"{int(row['total_tokens']):,}".replace(',', ' '),
                     f"{int(row['unique_words']):,}".replace(',', ' ')])
    add_table(doc, 'Таблица 1 – Характеристики частотных словарей',
              ['Словарь', 'Язык', 'Лемматизация', 'Токенов', 'Уникальных слов'],
              rows, widths=[6.0, 2.5, 3.0, 2.4, 2.8],
              align=[WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.CENTER,
                     WD_ALIGN_PARAGRAPH.CENTER, WD_ALIGN_PARAGRAPH.CENTER,
                     WD_ALIGN_PARAGRAPH.CENTER])

    add_para(doc, 'Полученные значения коэффициента c приведены в таблице 2.')
    rows = [[row['dictionary'], f"{int(round(row['c_median'])):,}".replace(',', ' ')]
            for _, row in zipf.iterrows()]
    add_table(doc, 'Таблица 2 – Коэффициент Ципфа c',
              ['Словарь', 'c'], rows, widths=[11.0, 5.0],
              align=[WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.CENTER])

    add_para(doc,
        'На рисунке 1 показаны графики для русского языка, на рисунке 2 — для английского. '
        'Верхние панели содержат эмпирические точки в логарифмических координатах и прямую '
        'f = c / r. Нижние панели показывают произведение f · r; горизонтальная линия '
        'соответствует коэффициенту c.')
    add_figure(doc, FIG / 'fig_zipf_ru.png',
               'Рисунок 1 – Закон Ципфа для русского языка', width_cm=16.5)
    add_figure(doc, FIG / 'fig_zipf_en.png',
               'Рисунок 2 – Закон Ципфа для английского языка', width_cm=16.5)

    ru_raw = zipf[zipf['key'] == 'ru_raw'].iloc[0]
    ru_lem = zipf[zipf['key'] == 'ru_lemma'].iloc[0]
    en_raw = zipf[zipf['key'] == 'en_raw'].iloc[0]
    en_lem = zipf[zipf['key'] == 'en_lemma'].iloc[0]
    add_para(doc,
        f"Для русского словаря без лемматизации c = {fmt_int(ru_raw['c_median'])}, "
        f"с лемматизацией — {fmt_int(ru_lem['c_median'])}. "
        f"Для английского словаря без лемматизации c = {fmt_int(en_raw['c_median'])}, "
        f"с лемматизацией — {fmt_int(en_lem['c_median'])}.")
    add_para(doc,
        'Во всех четырех словарях произведение f · r в средней части частотного списка '
        'остается приблизительно постоянным. На краях списка наблюдаются систематические '
        'отклонения: у самых частых слов произведение повышено, у редких — понижено.')
    add_para(doc,
        'Вывод по заданию 1: закон Ципфа для построенных частотных словарей выполняется '
        'приближенно. Коэффициент c зависит от языка и от способа подготовки словаря и '
        'составляет примерно 0,57–0,66 млн в рассматриваемом диапазоне рангов.')

    # ---------------- ЗАДАНИЕ 2 ----------------
    add_heading(doc, 'ЗАДАНИЕ 2. ЭМПИРИЧЕСКИЙ ЗАКОН ЦИПФА', page_break=True)
    add_para(doc,
        'Эмпирический закон Ципфа утверждает, что длина слова обратно пропорциональна его '
        'частоте: более короткие слова в среднем встречаются чаще длинных. Проверка выполнялась '
        'на списке служебных слов: предлогов, союзов, частиц, местоимений и вспомогательных глаголов.')
    add_para(doc,
        'Список служебных слов был составлен вручную и затем отфильтрован по наличию в '
        'соответствующем частотном словаре. Для каждого слова определялись длина L в символах '
        'и абсолютная частота f. Связь оценивалась коэффициентом корреляции Пирсона: '
        'отрицательное значение означает, что с увеличением длины слова частота в среднем убывает.')
    add_para(doc, 'Результаты корреляционного анализа приведены в таблице 3.')
    rows = [[row['dictionary'], int(row['n_function_words']), fmt(row['pearson_r'], 3)]
            for _, row in fw_summary.iterrows()]
    add_table(doc, 'Таблица 3 – Связь длины служебного слова и его частоты',
              ['Словарь', 'Служебных слов', 'r (Пирсон)'], rows,
              widths=[8.0, 4.0, 4.0],
              align=[WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.CENTER,
                     WD_ALIGN_PARAGRAPH.CENTER])

    add_para(doc,
        'На рисунке 3 показаны зависимости для русского языка, на рисунке 4 — для английского. '
        'По оси абсцисс отложена длина слова, по оси ординат — относительная частота.')
    add_figure(doc, FIG / 'fig_zipf2_ru.png',
               'Рисунок 3 – Частота и длина служебных слов русского языка', width_cm=16.5,
               page_break=True)
    add_figure(doc, FIG / 'fig_zipf2_en.png',
               'Рисунок 4 – Частота и длина служебных слов английского языка', width_cm=16.5)

    r_values = ', '.join(fmt(v, 3) for v in fw_summary['pearson_r'])
    add_para(doc,
        f"Во всех словарях коэффициенты корреляции отрицательные: r = {r_values}. "
        f"Это означает, что более короткие служебные слова в среднем встречаются чаще длинных.")
    add_para(doc,
        'При этом связь умеренная: длина слова объясняет лишь часть разброса частот. '
        'На частоту влияют также семантика, синтаксическая роль, омонимия и другие факторы.')
    add_para(doc,
        'Вывод по заданию 2: эмпирический закон Ципфа для служебных слов выполняется качественно. '
        'Связь длины и частоты отрицательная и статистически значимая, однако она не является '
        'строго обратно пропорциональной.')

    # ---------------- ЗАДАНИЕ 3 ----------------
    add_heading(doc, 'ЗАДАНИЕ 3. РАСЧЕТ КОЭФФИЦИЕНТА D ЖУЙАНА', page_break=True)
    add_para(doc,
        'Коэффициент D Жуйана характеризует равномерность распределения слова по разным сегментам '
        'корпуса. Для его расчета корпус разбивается на n равных сегментов, в каждом сегменте '
        'подсчитывается частота слова, после чего вычисляются средняя частота μ и среднее '
        'квадратическое отклонение σ:')
    add_formula(doc, 'D = 1 − σ / (μ · √(n − 1)),')
    add_para(doc,
        'где μ — средняя частота слова по сегментам; σ — среднее квадратическое отклонение частоты '
        'по сегментам (с делением на n); n — число сегментов.')
    add_para(doc,
        'Для расчета использовался корпус объемом 5 млн словоупотреблений, разбитый на 4 равных '
        'сегмента по 1,25 млн токенов. В каждом сегменте подсчитывались частоты слов из топ-1000 '
        'соответствующего частотного словаря. В таблицу 4 включены 10 самых частых слов каждого словаря.')
    add_para(doc, 'Результаты расчета коэффициента D приведены в таблице 4.')
    rows = [[row['dictionary'], row['word'], fmt(row['rel_freq_per_1e6'], 1),
             int(row['rank']), fmt(row['D'], 3)]
            for _, row in juilland_table.iterrows()]
    add_table(doc, 'Таблица 4 – Коэффициент D Жуйана для наиболее частых слов',
              ['Словарь', 'Слово', 'Отн. частота, ipm', 'Ранг', 'D'], rows,
              widths=[6.0, 3.0, 3.2, 1.6, 1.8],
              align=[WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.LEFT,
                     WD_ALIGN_PARAGRAPH.CENTER, WD_ALIGN_PARAGRAPH.CENTER,
                     WD_ALIGN_PARAGRAPH.CENTER], page_break=True)

    d_min = juilland_table['D'].min()
    d_max = juilland_table['D'].max()
    add_para(doc,
        f"Для десяти самых частых слов каждого словаря коэффициент D находится в диапазоне "
        f"от {fmt(d_min, 3)} до {fmt(d_max, 3)}. Это близко к единице: наиболее частотные слова "
        f"встречаются во всех сегментах корпуса примерно с одинаковой относительной частотой.")
    add_para(doc,
        'Вывод по заданию 3: коэффициент D Жуйана для наиболее частых слов близок к 1, что '
        'подтверждает равномерность их распределения по сегментам корпуса.')

    # ---------------- ЗАКЛЮЧЕНИЕ ----------------
    add_heading(doc, 'ЗАКЛЮЧЕНИЕ', page_break=True)
    add_para(doc,
        'В работе проведено статистическое исследование четырех частотных словарей, построенных '
        'по корпусу русской и английской Википедии: двух словарей без лемматизации и двух словарей '
        'с лемматизацией.')
    add_para(doc,
        'Проверка закона Ципфа показала, что произведение частоты на ранг приблизительно постоянно '
        'в средней части частотного списка. Коэффициент c составляет примерно 0,57–0,66 млн. '
        'На краях списка закон нарушается, что типично для реальных корпусов.')
    add_para(doc,
        'Проверка эмпирического закона Ципфа для служебных слов выявила статистически значимую '
        'отрицательную корреляцию между длиной слова и его частотой. Связь умеренная, поэтому '
        'закон выполняется качественно, а не как строгая обратная пропорциональность.')
    add_para(doc,
        'Расчет коэффициента D Жуйана показал, что для наиболее частых слов D близок к 1. '
        'Это соответствует равномерному распределению высокочастотной лексики по сегментам корпуса.')
    add_para(doc,
        'Таким образом, все три задания выполнены, расчеты воспроизводимы, а построенные графики '
        'и таблицы описывают статистические свойства полученных частотных словарей.')

    # ---------------- ИСТОЧНИКИ ----------------
    add_heading(doc, 'СПИСОК ИСПОЛЬЗОВАННЫХ ИСТОЧНИКОВ', page_break=True)
    sources = [
        '1. Задание на расчетно-графическую работу № 2. Статистические исследования на базе корпуса и частотных словарей. Минск, 2026.',
        '2. Wikimedia Foundation. Wikipedia. Dump 20231101 [Электронный ресурс]. URL: https://huggingface.co/datasets/wikimedia/wikipedia (дата обращения: 01.11.2026).',
        '3. Zipf G. K. Human Behavior and the Principle of Least Effort. Cambridge: Addison-Wesley, 1949. 573 p.',
        '4. Juilland A., Chang-Rodriguez E., Beltran J. V. Frequency Dictionary of Spanish Words. The Hague: Mouton, 1964.',
        '5. Manning C. D., Schütze H. Foundations of Statistical Natural Language Processing. Cambridge: MIT Press, 1999. 680 p.',
        '6. Баранов А. Н. Введение в прикладную лингвистику. М.: Эдиториал УРСС, 2001. 360 с.',
    ]
    for s in sources:
        p = doc.add_paragraph()
        pf = p.paragraph_format
        pf.first_line_indent = Cm(1.25)
        pf.line_spacing = Pt(18)
        pf.space_before = Pt(0)
        pf.space_after = Pt(0)
        pf.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
        r = p.add_run(s)
        set_run_font(r, size=14)

    doc.save(path)
    return path


In [4]:
def convert_to_pdf(docx_path):
    result = subprocess.run(
        ['soffice', '--headless', '--convert-to', 'pdf', '--outdir', str(REPORT_DIR), str(docx_path)],
        capture_output=True, text=True, timeout=180
    )
    if result.returncode != 0:
        print('LibreOffice stderr:', result.stderr)
        return None
    pdf_path = docx_path.with_suffix('.pdf')
    return pdf_path if pdf_path.exists() else None


def find_heading_pages(pdf_path, headings):
    txt = subprocess.check_output(['pdftotext', '-layout', str(pdf_path), '-'], text=True)
    pages = txt.split('\f')
    result = {}
    for h in headings:
        for i, page in enumerate(pages, start=1):
            if i <= 2:
                continue
            normalized = re.sub(r'\s+', ' ', page)
            if h in normalized:
                result[h] = i
                break
    return result

draft_path = REPORT_DIR / 'РГР_2_отчет.docx'
build_report(draft_path, pages=None)
print('Черновик сохранен:', draft_path)

pdf_draft = convert_to_pdf(draft_path)
pages = {}
if pdf_draft:
    pages = find_heading_pages(pdf_draft, TOC_ENTRIES)
    print('Найденные страницы разделов:', pages)
else:
    print('Не удалось получить PDF для определения номеров страниц.')

final_path = REPORT_DIR / 'РГР_2_отчет.docx'
build_report(final_path, pages=pages)
print('Итоговый DOCX сохранен:', final_path)

pdf_final = convert_to_pdf(final_path)
if pdf_final:
    print('PDF для проверки:', pdf_final)
else:
    print('PDF не создан, но DOCX готов.')


Черновик сохранен: /home/dd/nlp_uni/rgr2/report/РГР_2_отчет.docx


Найденные страницы разделов: {'ВВЕДЕНИЕ': 3, 'ЗАДАНИЕ 1. ЗАКОН ЦИПФА': 4, 'ЗАДАНИЕ 2. ЭМПИРИЧЕСКИЙ ЗАКОН ЦИПФА': 7, 'ЗАДАНИЕ 3. РАСЧЕТ КОЭФФИЦИЕНТА D ЖУЙАНА': 9, 'ЗАКЛЮЧЕНИЕ': 11, 'СПИСОК ИСПОЛЬЗОВАННЫХ ИСТОЧНИКОВ': 12}


Итоговый DOCX сохранен: /home/dd/nlp_uni/rgr2/report/РГР_2_отчет.docx


PDF для проверки: /home/dd/nlp_uni/rgr2/report/РГР_2_отчет.pdf


## Что нужно проверить вручную

1. Заменить на титульном листе `[ФИО студента]`, `[курс]` и `[группа]` на свои данные (это можно сделать в Word или изменить переменные в начале ноутбука и перезапустить его).
2. Открыть `rgr2/report/РГР_2_отчет.docx` в Word и при необходимости обновить поля (номера страниц). Оглавление уже заполнено номерами страниц, найденными по PDF.
3. Проверить, что на компьютере установлен шрифт Times New Roman. В DOCX он указан явно; при отсутствии шрифта Word подставит другой.
